# Technology Classification As Baseline ᯓ★🎸
![Electric Guitar - Give me the base](electric_guitar.JPG)
<p style="font-size: 0.8em; margin-top: 4px;">
  <a href="https://unsplash.com/de/fotos/weisse-und-schwarze-e-gitarre-0qUEcUoJmlk" target="_blank">Photo by David Guliciuc</a>
</p>

## Purpose

This notebook establishes a **baseline model** to quantify how much emission 
variance can be explained by production technology alone. This serves as a 
foundation for evaluating more complex models that incorporate external drivers 
(carbon pricing, policy) and internal factors (corporate commitments).

## From EDA to Baseline

Our exploratory analysis revealed a clear technology gap:
- **EAF** (Electric Arc Furnace): ~0.2-1.2 tCO₂e/t
- **BF-BOF** (Blast Furnace): ~1.5-2.5 tCO₂e/t

Now we quantify this statistically: **How much variance does technology explain?**

## Model Approach

**Type:** Simple decision tree (binary classification)
- **Predictor:** Technology type (EAF vs. BF-BOF)
- **Outcome:** Operational emission intensity (Scope 1+2)
- **Validation:** Statistical significance (t-test) + variance explained (R²)

**Why this matters:** Technology likely explains the majority of emission variance 
across steel producers. Understanding this baseline helps us set realistic expectations 
for what external factors (pricing, policy) and internal factors (commitments) can 
potentially explain - if technology accounts for most variance, other factors can 
only influence the remainder.

## 1. Setup & Data Loading

In [ ]:
# Setup
import sys
from pathlib import Path

# Add scripts folder to path
scripts_path = Path.cwd().parent / 'scripts'
sys.path.insert(0, str(scripts_path))

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# Import custom data loader
from data_loader import (
    load_company_data,
    load_eu_data,
    load_global_data,
    filter_complete_data,
    prepare_analysis_dataset,
    print_data_summary
)

# Import custom plotting module
from plotting_utils import plot_technology_boxplot


In [ ]:
# Load and prepare data using data_loader
df_raw = load_company_data(fix_apostrophes=True, filter_region='Europe')
df_filtered = filter_complete_data(df_raw, min_years=4)
df_analysis = prepare_analysis_dataset(df_filtered)
print_data_summary(df_analysis)

# Load EU and global context data
eu_data = load_eu_data()
global_data = load_global_data()
print(f"✓ EU benchmark data loaded")
print(f"✓ Global benchmark data loaded")

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

## 2. Statistical Analysis

### 2.1 Descriptive Statistics

In [ ]:
# Calculate statistics by technology
tech_stats = df_analysis.groupby('technology_group')['total_intensity_best'].agg([
    ('N', 'count'),
    ('Mean', 'mean'),
    ('Std', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(3)

print("📊 Descriptive Statistics by Technology:\n")
print(tech_stats.to_string())

# Calculate key metrics
# Calculate key metrics
eaf_data = df_analysis[df_analysis['technology_group'] == 'EAF']['total_intensity_best']
bof_data = df_analysis[df_analysis['technology_group'] == 'BF-BOF']['total_intensity_best']
eaf_mean = eaf_data.mean()
bof_mean = bof_data.mean()
difference = bof_mean - eaf_mean
pct_lower = (difference / bof_mean) * 100

print(f"\n🔑 Key Findings:")
print(f"   EAF average:    {eaf_mean:.3f} tCO₂e/t")
print(f"   BF-BOF average: {bof_mean:.3f} tCO₂e/t")
print(f"   Difference:     {difference:.3f} tCO₂e/t ({pct_lower:.1f}% lower for EAF)")

In [ ]:
fig, ax, _ = plot_technology_boxplot(
    df_analysis,
    intensity_col='total_intensity_best',
    ylabel='Operational Intensity (Scope 1+2, tCO$_2$e/t)',
    show_stats=True,
    save_path='../results/models/baseline_tech_comparison.png'
)

### 2.2 Statistical Significance Test (t-test)

**Null Hypothesis (H₀):** No difference in emissions between EAF and BF-BOF  
**Alternative Hypothesis (H₁):** EAF has significantly lower emissions than BF-BOF

We use an independent samples t-test to determine if observed differences are statistically significant.

In [ ]:
# Separate data by technology
eaf_data = df_analysis[df_analysis['technology_group'] == 'EAF']['total_intensity_best']
bof_data = df_analysis[df_analysis['technology_group'] == 'BF-BOF']['total_intensity_best']

# Perform t-test
t_stat, p_value = stats.ttest_ind(eaf_data, bof_data)

print(f"📊 Independent Samples T-Test:")
print(f"   t-statistic: {t_stat:.4f}")
print(f"   p-value:     {p_value:.6f}")

# Interpret significance
if p_value < 0.001:
    sig_level = "Highly significant (p < 0.001) ***"
elif p_value < 0.01:
    sig_level = "Significant (p < 0.01) **"
elif p_value < 0.05:
    sig_level = "Significant (p < 0.05) *"
else:
    sig_level = "Not significant (p ≥ 0.05)"

print(f"   Result:      {sig_level}")
print(f"\n✅ Conclusion: Technology differences are {sig_level.lower()}")

### 2.3 Model Performance (R²)

Calculate how much variance in emissions the baseline model explains.

In [ ]:
# Create predictions using group means
df_analysis['predicted'] = df_analysis['technology_group'].map({
    'EAF': eaf_mean,
    'BF-BOF': bof_mean
})
df_analysis['residual'] = df_analysis['total_intensity_best'] - df_analysis['predicted']

# Calculate R²
y_mean = df_analysis['total_intensity_best'].mean()
SS_total = ((df_analysis['total_intensity_best'] - y_mean) ** 2).sum()
SS_residual = ((df_analysis['residual']) ** 2).sum()
r_squared = 1 - (SS_residual / SS_total)

# Calculate error metrics
mae = np.abs(df_analysis['residual']).mean()
rmse = np.sqrt((df_analysis['residual'] ** 2).mean())

print(f"📊 Model Performance:")
print(f"   R² (variance explained): {r_squared:.4f} ({r_squared*100:.1f}%)")
print(f"   MAE:  {mae:.3f} tCO₂e/t")
print(f"   RMSE: {rmse:.3f} tCO₂e/t")

print(f"\n✅ Baseline model explains {r_squared*100:.1f}% of emission variance using technology alone")

### 2.4 Model Validation with sklearn

We validate our manual calculations using sklearn's DecisionTreeRegressor to ensure reproducibility and confirm the baseline model's performance.

In [ ]:
### Same analysis using sklearn's built-in functions

# Encode technology as binary (0 = EAF, 1 = BF-BOF)
df_analysis['tech_encoded'] = (df_analysis['technology_group'] == 'BF-BOF').astype(int)

# Fit decision tree
X = df_analysis[['tech_encoded']]
y = df_analysis['total_intensity_best']

model = DecisionTreeRegressor(max_depth=1, random_state=42)
model.fit(X, y)

# Calculate metrics
y_pred = model.predict(X)
r_squared = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = root_mean_squared_error(y, y_pred)

print("✓ sklearn validation:")
print(f"  R² = {r_squared:.3f} ({r_squared*100:.1f}% variance explained)")
print(f"  MAE = {mae:.3f} tCO₂e/t")
print(f"  RMSE = {rmse:.3f} tCO₂e/t")

## 3. Interpretation & Implications

### 3.1 Model Performance

**Statistical Validation:**
- ✅ **Highly significant difference** (t-test: p < 0.001***)
- ✅ **Strong explanatory power** (R² = 0.798, explaining ca. 80% of variance)
- ✅ **Low prediction error** (MAE = 0.24 tCO₂e/t, RMSE = 0.30 tCO₂e/t)

**Technology Gap:**
- EAF average: 0.61 tCO₂e/t
- BF-BOF average: 1.81 tCO₂e/t  
- Difference: 1.20 tCO₂e/t (67% lower emissions for EAF)

### 3.2 What This Means

**Technology Dominates Emissions**

Production technology explains 80% of emission variance across European steel producers. This means:
- Technology choice is the primary driver of emission performance
- External factors (carbon pricing, policy) can influence at most 20% of variance
- Internal factors (commitments, investments) can influence at most 20% of variance

The 1.20 tCO₂e/t technology gap between EAF and BF-BOF represents the fundamental difference in production processes - electric arc furnaces using scrap metal versus blast furnaces using iron ore and coal.

**Implications for Decarbonization**

The strong technology signal suggests:
- ✅ **Technology transitions** (BF-BOF → EAF or H₂-DRI) offer the largest emission reductions
- ⚠️ **Incremental improvements** within existing technologies have limited impact (~20% maximum)
- ⚠️ **Carbon pricing alone** cannot close the 1.20 tCO₂e/t technology gap without enabling technology transitions


### 3.3 Next Steps

This baseline model establishes the foundation for:
1. **External drivers analysis**: Do carbon prices, policy changes drive improvements 
   within technology groups?
2. **Internal factors analysis**: Do corporate commitments correlate with better 
   performance relative to technology peers?
3. **Action score development**: Combining quantitative trends with qualitative 
   sustainability reporting analysis

When we add more features to the analysis, we should expect modest improvements in explanatory power (R² from 0.80 to perhaps 0.85-0.90), not dramatic changes.


## 4. Export Clean Dataset

Save the prepared dataset for use in subsequent analyses.

In [ ]:
# Select key columns for export
export_cols = ['company', 'country', 'technology', 'technology_group', 'year', 
               'production', 'total_intensity_best', 'scope2_method_used', 
               'predicted', 'residual']

data_export = df_analysis[export_cols].copy()

# Save to CSV
data_export.to_csv('../results/models/baseline_model_dataset.csv', index=False)

print(f"✅ Clean dataset exported: {len(data_export)} rows, {len(export_cols)} columns")
print(f"   Saved to: ../results/models/baseline_model_dataset.csv")